In [1]:
import pandas as pd
import spacy
from date_spacy import find_dates

!spacy download en_core_web_sm

In [2]:
# Load the spaCy model for Named Entity Recognition (NER)
nlp = spacy.load("en_core_web_sm")

# Add Social Security Number
ssn_pattern_regex = {
    "label": "SSN",
    "pattern": [
        {"TEXT": {"REGEX": "\\d{3}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{2}"}},
        {"TEXT": "-"},
        {"TEXT": {"REGEX": "\\d{4}"}}
    ]
}
ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True}, before="ner")
ruler.add_patterns([ssn_pattern_regex])

# Add gender/sex pattern
gender_pattern_regex = {
    "label": "GENDER",
    "pattern": [
        {"LOWER": {"REGEX": "\\b(gender|sex|male|female|man|woman|boy|girl|he|she|him|her)\\b"}}
    ]
}
ruler.add_patterns([gender_pattern_regex])

# Add date finder
nlp.add_pipe('find_dates', after="ner")

# Add specific date patterns
# date_pattern_regex = {
#     "label": "DATE",
#     "pattern": [
#         {"TEXT": {"REGEX": "\\d{2}/\\d{2}/\\d{4}"}},
#         {"TEXT": {"REGEX": "\\d{1,2}\\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\\s+\\d{4}"}}
#     ]
# }
# ruler.add_patterns([date_pattern_regex])


<function date_spacy.components.find_dates(doc)>

In [3]:
# Sample dataframe
data = {'comments': [
    "My credit card number is 1234-5678-9012-3456",
    "My date of birth is 01/01/1990",
    "My driver's license number is A1234567",
    "My financial information includes bank account number 123456789",
    "The full name is John Doe",
    "My gender is male",
    "My mailing address is 123 Main St, Anytown, USA",
    "My medical records show I have diabetes",
    "My passport information is passport number 987654321",
    "My place of birth is Anytown, USA",
    "My race is Caucasian",
    "My religion is Christianity",
    "My Social Security number (SSN) is 123-45-6789",
    "My ZIP code is 12345"
]}


In [4]:
df = pd.DataFrame(data)
df

,comments
0,My credit card number is 1234-5678-9012-3456
1,My date of birth is 01/01/1990
2,My driver's license number is A1234567
3,My financial information includes bank account...
4,The full name is John Doe
5,My gender is male
6,"My mailing address is 123 Main St, Anytown, USA"
7,My medical records show I have diabetes
8,My passport information is passport number 987...
9,"My place of birth is Anytown, USA"


In [5]:
# Define a function to redact PII using spaCy NER
def redact_pii(text):
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["SSN", 
                          "GENDER", 
                          "PERSON", 
                          "NORP", 
                          "FAC", 
                        #   "ORG", 
                        #   "GPE", 
                          "LOC", 
                          "PRODUCT", 
                          "EVENT", 
                          "WORK_OF_ART", 
                          "LAW", 
                          "LANGUAGE", 
                          "DATE", 
                          "TIME", 
                          "PERCENT", 
                          "MONEY", 
                          "QUANTITY", 
                          "ORDINAL", 
                          "CARDINAL"]:
            redacted_text = redacted_text.replace(ent.text, "[REDACTED]")
        else:
            redacted_text = redacted_text.replace(ent.text, "[REDACTED]")
    return redacted_text


In [6]:
# Apply the function to the comments column
df['comments'] = df['comments'].apply(redact_pii)

df

,comments
0,My credit card number is [REDACTED]-3456
1,My date of birth is [REDACTED]
2,My driver's license number is A1234567
3,My financial information includes bank account...
4,The full name is [REDACTED]
5,My [REDACTED] is [REDACTED]
6,"My mailing address is [REDACTED] [REDACTED], [..."
7,My medical records show I have diabetes
8,My passport information is passport number [RE...
9,My place of birth is [REDACTED]


In [19]:
doc = nlp("year quarter, The social Security number (SSN) is 123-45-6789 and I love AI . name is limm kitsiang the cto, Educate and train personnel before actually using them.")
for ent in doc.ents:
    print(ent.text, ent.label_)

year quarter DATE
SSN ORG
123-45-6789 SSN
AI GPE
Educate ORG
